In [2]:
!pip install python-docx pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 8.1 MB/s eta 0:00:00


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.datasets import STL10
from torchvision import transforms, models
import pandas as pd
import numpy as np
from docx import Document

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device =>", device)

Device => cuda


In [4]:
DATA_ROOT = "/content/data"

test_tf = transforms.Compose([
    transforms.Resize((96,96)),
    transforms.ToTensor()
])

test_set = STL10(root=DATA_ROOT, split="test", download=True, transform=test_tf)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False)
len(test_set)


8000

In [5]:
# UPDATE path
CKPT = "/content/drive/MyDrive/ssl_checkpoints/byol_epoch_200.pth"

def load_byol_encoder():
    model = models.resnet18(weights=None)
    model.fc = nn.Identity()
    state = torch.load(CKPT, map_location=device)
    model.load_state_dict(state)
    return model.to(device)

encoder = load_byol_encoder()
encoder.eval()
print("Encoder Loaded!")


Encoder Loaded!


In [6]:
def linear_eval(encoder, loader):
    encoder.eval()
    clf = nn.Linear(512, 10).to(device)
    opt = torch.optim.Adam(clf.parameters(), lr=3e-3)
    loss_fn = nn.CrossEntropyLoss()

    # train
    for epoch in range(3):
        for x,y in loader:
            x,y = x.to(device), y.to(device)
            with torch.no_grad(): feats = encoder(x)
            loss = loss_fn(clf(feats), y)
            opt.zero_grad(); loss.backward(); opt.step()

    # test accuracy
    correct = total = 0
    with torch.no_grad():
        for x,y in loader:
            x,y = x.to(device), y.to(device)
            feats = encoder(x)
            pred = clf(feats).argmax(1)
            correct += (pred==y).sum().item(); total+=y.size(0)

    return round(100 * correct / total, 2)


In [7]:
noaug_tf = transforms.Compose([
    transforms.Resize((96,96)),
    transforms.ToTensor()
])
noaug_loader = DataLoader(STL10(DATA_ROOT, split="train", download=False, transform=noaug_tf),
                          batch_size=256, shuffle=False)

acc_clean = linear_eval(encoder, test_loader)
acc_noaug = linear_eval(encoder, noaug_loader)

print("BASELINE:", acc_clean)
print("Without Augmentations:", acc_noaug)


BASELINE: 79.75
Without Augmentations: 79.38


In [8]:
def add_noise(x, std=0.2):
    return x + torch.randn_like(x) * std

noise_tf = transforms.Compose([
    transforms.Resize((96,96)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: add_noise(x))
])
noise_loader = DataLoader(STL10(DATA_ROOT, split="test", transform=noise_tf),
                          batch_size=256, shuffle=False)

acc_noise = linear_eval(encoder, noise_loader)
print("Noise accuracy:", acc_noise)


Noise accuracy: 65.75


In [9]:
def blackout(x, size=24):
    x = x.clone()
    x[:, :size, :size] = 0
    return x

occ_tf = transforms.Compose([
    transforms.Resize((96,96)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: blackout(x))
])
occ_loader = DataLoader(STL10(DATA_ROOT, split="test", transform=occ_tf),
                        batch_size=256, shuffle=False)

acc_occ = linear_eval(encoder, occ_loader)
print("Occlusion accuracy:", acc_occ)


Occlusion accuracy: 78.69


In [10]:
feat_results = {}

for dim in [64, 128, 256, 512]:
    clf = nn.Linear(512, dim).to(device)
    proj = nn.Linear(dim, 10).to(device)

    def classifier(feats):
        h = clf(feats)
        return proj(h)

    # train classifier
    opt = torch.optim.Adam(list(clf.parameters())+list(proj.parameters()), lr=3e-3)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(2):
        for x,y in test_loader:
            x,y=x.to(device),y.to(device)
            with torch.no_grad(): f = encoder(x)
            loss=loss_fn(classifier(f),y)
            opt.zero_grad(); loss.backward(); opt.step()

    # accuracy
    correct = total = 0
    with torch.no_grad():
        for x,y in test_loader:
            x,y=x.to(device),y.to(device)
            f=encoder(x)
            pred = classifier(f).argmax(1)
            correct +=(pred==y).sum().item(); total+=y.size(0)

    feat_results[f"dim={dim}"] = round(100*correct/total,2)

feat_results


{'dim=64': 79.42, 'dim=128': 79.17, 'dim=256': 80.1, 'dim=512': 79.24}

In [11]:
df = pd.DataFrame({
    "Test": ["Baseline", "No Augment", "Noise", "Occlusion"] + list(feat_results.keys()),
    "Accuracy (%)": [acc_clean, acc_noaug, acc_noise, acc_occ] + list(feat_results.values())
})
df


,Test,Accuracy (%)
0,Baseline,79.75
1,No Augment,79.38
2,Noise,65.75
3,Occlusion,78.69
4,dim=64,79.42
5,dim=128,79.17
6,dim=256,80.10
7,dim=512,79.24


In [12]:
doc = Document()
doc.add_heading("BYOL Ablation Study", level=1)
doc.add_paragraph("Automatically generated ablation table.")

table = doc.add_table(rows=df.shape[0]+1, cols=df.shape[1])
for j,col in enumerate(df.columns):
    table.cell(0,j).text = col

for i,row in df.iterrows():
    for j,val in enumerate(row):
        table.cell(i+1,j).text = str(val)

SAVE_FILE = "/content/ablation_results.docx"
doc.save(SAVE_FILE)
SAVE_FILE


'/content/ablation_results.docx'

In [13]:
from google.colab import files
files.download(SAVE_FILE)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>